In [ ]:
#!/usr/bin/env python3
"""
Complete script for LPG infrastructure detection in sub‑Saharan Africa.
Combines OpenStreetMap (Overpass) and Google Places APIs.
Outputs: CSV, JSON, GeoPackage (EPSG:3857) + summary CSV.
Resume capability via processed_points.txt.
"""

import os
import sys
import time
import json
import csv
import math
import random
import requests
from collections import Counter, defaultdict
from datetime import datetime

import geopandas as gpd
import pandas as pd
from shapely.geometry import Point, Polygon, box
from shapely.ops import unary_union

# ======================== CONFIGURATION ========================

USE_OSM = True                     # Set to False to skip OpenStreetMap queries
USE_GOOGLE = False                  # PAY ATTENTION TO COST; Set to False to skip Google Places queries
GOOGLE_API_KEY = "YOUR_VALID_API_KEY"   # Insert your Google Places API key
STEP = 0.65                         # Grid spacing in degrees (~72 km)
RADIUS = 50000                      # Search radius in metres
RESUME = True                       # If True, skip points already processed

# Paths
OUTPUT_ROOT = "location_total"      # all outputs go here
WORLD_SHP_PATH = os.path.join(OUTPUT_ROOT, "world_bound", "ne_110m_admin_0_countries.shp")
PROCESSED_POINTS_FILE = os.path.join(OUTPUT_ROOT, "processed_points.txt")

TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

# ======================== COUNTRY DEFINITIONS ========================

SSA_COUNTRIES = {
    'angola':               {'languages': ['en', 'pt']},
    'benin':                {'languages': ['en', 'fr']},
    'botswana':             {'languages': ['en']},
    'burkina_faso':         {'languages': ['en', 'fr']},
    'burundi':              {'languages': ['en', 'fr', 'sw']},
    'cameroon':             {'languages': ['en', 'fr']},
    'central_african_republic': {'languages': ['en', 'fr']},
    'chad':                 {'languages': ['en', 'fr', 'ar']},
    'congo':                {'languages': ['en', 'fr']},
    'congo_drc':            {'languages': ['en', 'fr', 'sw']},
    'cote_divoire':         {'languages': ['en', 'fr']},
    'djibouti':             {'languages': ['en', 'fr', 'ar']},
    'equatorial_guinea':    {'languages': ['en', 'pt', 'fr']},
    'eritrea':              {'languages': ['en', 'ar']},
    'eswatini':             {'languages': ['en']},
    'ethiopia':             {'languages': ['en']},
    'gabon':                {'languages': ['en', 'fr']},
    'gambia':               {'languages': ['en']},
    'ghana':                {'languages': ['en']},
    'guinea':               {'languages': ['en', 'fr']},
    'guinea_bissau':        {'languages': ['en', 'pt']},
    'kenya':                {'languages': ['en', 'sw']},
    'lesotho':              {'languages': ['en']},
    'liberia':              {'languages': ['en']},
    'madagascar':           {'languages': ['en', 'fr']},
    'malawi':               {'languages': ['en']},
    'mali':                 {'languages': ['en', 'fr']},
    'mauritania':           {'languages': ['en', 'fr', 'ar']},
    'mozambique':           {'languages': ['en', 'pt']},
    'namibia':              {'languages': ['en']},
    'niger':                {'languages': ['en', 'fr']},
    'nigeria':              {'languages': ['en']},
    'rwanda':               {'languages': ['en', 'fr', 'sw']},
    'senegal':              {'languages': ['en', 'fr']},
    'sierra_leone':         {'languages': ['en']},
    'somalia':              {'languages': ['en', 'ar']},
    'south_africa':         {'languages': ['en']},
    'south_sudan':          {'languages': ['en', 'ar', 'sw']},
    'sudan':                {'languages': ['en', 'ar']},
    'tanzania':             {'languages': ['en', 'sw']},
    'togo':                 {'languages': ['en', 'fr']},
    'uganda':               {'languages': ['en', 'sw']},
    'zambia':               {'languages': ['en']},
    'zimbabwe':             {'languages': ['en']}
}

# ======================== KEYWORD DICTIONARIES ========================

RESELLER_KEYWORDS = {
    'en': [
        "LPG refill station", "gas station with LPG", "LPG filling point",
        "butane gas", "LPG dealer", "gas cylinder exchange",
        "cooking gas", "LPG"
    ],
    'fr': [
        "station de remplissage de GPL", "station-service avec GPL",
        "point de remplissage de GPL",
        "revendeur de GPL", "échange de bouteilles de gaz",
        "gaz de cuisine", "GPL", "gaz butane"
    ],
    'pt': [
        "estação de recarga de GLP", "posto de gasolina com GLP",
        "gás butano", "ponto de enchimento de GLP",
        "revendedor de GLP", "troca de botijão de gás",
        "gás de cozinha", "GLP"
    ],
    'sw': [
        "kituo cha kujaza LPG", "kituo cha mafuta chenye LPG",
        "sehemu ya kujaza LPG", "muuzaji wa LPG",
        "kubadilisha mitungi ya gesi", "gesi ya butane",
        "gesi ya kupikia", "LPG"
    ],
    'ar': [
        "محطة تعبئة غاز البترول المسال", "محطة وقود مع غاز البترول المسال",
        "غاز البيوتان", "نقطة تعبئة غاز البترول المسال",
        "تاجر غاز البترول المسال", "استبدال اسطوانات الغاز",
        "غاز الطبخ", "غاز البترول المسال"
    ]
}

FILLING_KEYWORDS = {
    'en': [
        "LPG cylinder filling plant", "LPG bottling plant",
        "gas bottling plant", "LPG filling plant",
        "cylinder filling station", "LPG bottling station"
    ],
    'fr': [
        "usine de remplissage de bouteilles de GPL", "usine d'embouteillage de GPL",
        "usine d'embouteillage de gaz", "usine de remplissage de GPL",
        "station de remplissage de bouteilles", "station d'embouteillage de GPL"
    ],
    'pt': [
        "fábrica de enchimento de botijas de GLP", "fábrica de engarrafamento de GLP",
        "fábrica de engarrafamento de gás", "fábrica de enchimento de GLP",
        "estação de enchimento de botijas", "estação de engarrafamento de GLP"
    ],
    'sw': [
        "kiwanda cha kujaza mitungi ya LPG",
        "kiwanda cha kujaza LPG",
        "kiwanda cha kujaza gesi", "kiwanda cha kusindika LPG",
        "kituo cha kujaza mitungi", "kituo cha kujaza LPG"
    ],
    'ar': [
        "مصنع تعبئة اسطوانات غاز البترول المسال", "مصنع تعبئة غاز البترول المسال",
        "مصنع تعبئة الغاز", "محطة تعبئة اسطوانات الغاز",
        "محطة تعبئة غاز البترول المسال", "محطة تعبئة الغاز"
    ]
}

# ======================== HELPER FUNCTIONS ========================

def load_country_geometries(ssa_dict, shp_path):
    """Load world boundaries and filter to sub‑Saharan African countries.
    Returns dict: country_lower -> GeoDataFrame row."""
    world = gpd.read_file(shp_path)
    # Normalise names: try 'ADMIN' field (common in Natural Earth)
    possible_name_fields = ['ADMIN', 'NAME', 'SOVEREIGNT']
    name_col = None
    for col in possible_name_fields:
        if col in world.columns:
            name_col = col
            break
    if name_col is None:
        raise KeyError("No recognised country name column found in shapefile.")
    world['name_lower'] = world[name_col].str.lower()
    ssa_names = set(ssa_dict.keys())
    ssa_gdf = world[world['name_lower'].isin(ssa_names)].copy()
    if ssa_gdf.empty:
        raise ValueError("No matching countries found. Check shapefile naming.")
    # Map name to row
    country_rows = {}
    for _, row in ssa_gdf.iterrows():
        country_rows[row['name_lower']] = row
    return country_rows, world.crs

def buffer_geometry(geom, buffer_deg):
    """Buffer a geometry in degrees (approximate)."""
    return geom.buffer(buffer_deg)

def generate_grid(geom, step):
    """Generate regular grid points inside a geometry.
    Returns list of (lat, lon) tuples."""
    minx, miny, maxx, maxy = geom.bounds
    points = []
    lat = miny
    while lat <= maxy:
        lon = minx
        while lon <= maxx:
            pt = Point(lon, lat)
            if geom.contains(pt):
                points.append((round(lat, 6), round(lon, 6)))  # keep high precision for containment
            lon += step
        lat += step
    return points

def deduplicate_grid_points(country_points):
    """Takes list of (lat, lon, country_name) tuples.
    Merges points with same (lat, lon) rounded to 4 decimals and returns
    list of (lat, lon, set_of_countries)."""
    merged = defaultdict(set)
    for lat, lon, country in country_points:
        key = (round(lat, 4), round(lon, 4))
        merged[key].add(country)
    result = [(key[0], key[1], countries) for key, countries in merged.items()]
    return result

def languages_for_countries(countries):
    """Return union of languages for a set of country names."""
    langs = set()
    for c in countries:
        if c in SSA_COUNTRIES:
            langs.update(SSA_COUNTRIES[c]['languages'])
    return langs

def osm_query_bbox(bbox, max_retries=3):
    """Query Overpass API for a bounding box (min_lat, max_lat, min_lon, max_lon).
    Returns list of place dicts."""
    min_lat, max_lat, min_lon, max_lon = bbox
    query = f"""
    [out:json];
    (
        node["amenity"="fuel"]["fuel:lpg"~"yes|only|lpg|LPG|GPL|GLP"]({min_lat},{min_lon},{max_lat},{max_lon});
        way["amenity"="fuel"]["fuel:lpg"~"yes|only|lpg|LPG|GPL|GLP"]({min_lat},{min_lon},{max_lat},{max_lon});
        node["shop"="gas"]({min_lat},{min_lon},{max_lat},{max_lon});
        way["shop"="gas"]({min_lat},{min_lon},{max_lat},{max_lon});
        node["shop"="gas_cooking"]({min_lat},{min_lon},{max_lat},{max_lon});
        way["shop"="gas_cooking"]({min_lat},{min_lon},{max_lat},{max_lon});
        node["industrial"="gas"]({min_lat},{min_lon},{max_lat},{max_lon});
        node["man_made"="storage_tank"]["content"~"lpg|gpl|glp|gas|butane|propane"]({min_lat},{min_lon},{max_lat},{max_lon});
        way["industrial"="gas"]({min_lat},{min_lon},{max_lat},{max_lon});
        way["man_made"="storage_tank"]["content"~"lpg|gpl|glp|gas|butane|propane"]({min_lat},{min_lon},{max_lat},{max_lon});
    );
    out center;
    """
    url = "https://overpass-api.de/api/interpreter"
    headers = {
        'User-Agent': 'LPG-Research-Script/1.0 (contact: your-email@example.com)',
        'Accept': 'application/json'
    }
    for attempt in range(max_retries):
        try:
            resp = requests.post(url, data={'data': query}, headers=headers, timeout=60)
            if resp.status_code == 200:
                data = resp.json()
                break
            elif resp.status_code in [429, 504]:
                wait = (2 ** attempt) + random.uniform(0, 1)
                print(f"    OSM bbox {resp.status_code} - retry in {wait:.1f}s (attempt {attempt+1}/{max_retries})")
                time.sleep(wait)
            else:
                return [], resp.status_code
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2)
            else:
                return [], f"Exception({e})"
    else:
        return [], resp.status_code

    results = []
    for elem in data.get('elements', []):
        if elem['type'] == 'node':
            lat_place = elem.get('lat')
            lon_place = elem.get('lon')
        else:
            center = elem.get('center', {})
            lat_place = center.get('lat')
            lon_place = center.get('lon')
        if lat_place is None or lon_place is None:
            continue
        tags = elem.get('tags', {})
        if 'amenity' in tags and tags['amenity'] == 'fuel':
            ptype = 'fuel_station'
        elif 'shop' in tags:
            ptype = tags['shop']
        else:
            ptype = 'lpg_related'

        is_fuel_station = tags.get('amenity') == 'fuel' and (
            tags.get('fuel:lpg') or tags.get('fuel:GPL') or tags.get('fuel:lpg') in ['yes', 'only']
        )
        is_storage_tank = tags.get('man_made') == 'storage_tank' and tags.get('content', '').lower() in ['lpg', 'gpl', 'glp', 'gas', 'butane', 'propane']
        is_industrial_gas = tags.get('industrial') == 'gas'
        category = 'filling' if (is_fuel_station or is_storage_tank or is_industrial_gas) else 'reseller'

        name = tags.get('name', '')
        results.append({
            'place_id': f"osm_{elem['id']}",
            'name': name,
            'address': '',
            'lat': lat_place,
            'lng': lon_place,
            'types': ptype,
            'keyword': 'osm',
            'search_lat': (min_lat + max_lat) / 2,
            'search_lon': (min_lon + max_lon) / 2,
            'rating': None,
            'user_ratings_total': 0,
            'source': 'osm',
            'language': '',
            'category': category
        })
    return results, None

def osm_search_adaptive(lat, lon, radius, min_side_km=10):
    """Adaptive OSM search: starts with a bbox derived from radius;
    if the query fails, recursively splits the box until min side size."""
    # Convert radius to approximate degrees (latitude degrees)
    deg_per_km = 1 / 111.0
    half_deg = radius * deg_per_km / 1000.0  # radius in km -> deg
    min_lat = lat - half_deg
    max_lat = lat + half_deg
    min_lon = lon - half_deg / math.cos(math.radians(lat))
    max_lon = lon + half_deg / math.cos(math.radians(lat))
    bbox = (min_lat, max_lat, min_lon, max_lon)

    def recursive_query(bbox, depth=0):
        # Try direct query
        results, err = osm_query_bbox(bbox)
        if err is None:
            return results
        # If error and bbox side length > min_side, split
        (min_lat, max_lat, min_lon, max_lon) = bbox
        lat_span = max_lat - min_lat
        lon_span = max_lon - min_lon
        min_lat_km = lat_span * 111
        min_lon_km = lon_span * 111 * math.cos(math.radians((min_lat + max_lat) / 2))
        if min_lat_km <= min_side_km or min_lon_km <= min_side_km or depth > 5:
            return []  # give up
        # Split into 4
        mid_lat = (min_lat + max_lat) / 2
        mid_lon = (min_lon + max_lon) / 2
        all_res = []
        for sub in [
            (min_lat, mid_lat, min_lon, mid_lon),
            (min_lat, mid_lat, mid_lon, max_lon),
            (mid_lat, max_lat, min_lon, mid_lon),
            (mid_lat, max_lat, mid_lon, max_lon)
        ]:
            time.sleep(1)  # OSM courtesy
            sub_res = recursive_query(sub, depth+1)
            all_res.extend(sub_res)
        # deduplicate by place_id
        seen = {}
        for p in all_res:
            pid = p['place_id']
            if pid not in seen:
                seen[pid] = p
        return list(seen.values())

    return recursive_query(bbox)

def google_nearby_search(lat, lon, keyword, lang, category):
    """Perform a Google Places Nearby Search with pagination."""
    url = "https://maps.googleapis.com/maps/api/place/nearbysearch/json"
    results = []
    params = {
        'location': f"{lat},{lon}",
        'radius': RADIUS,
        'keyword': keyword,
        'language': lang,
        'key': GOOGLE_API_KEY
    }
    while True:
        try:
            resp = requests.get(url, params=params)
            data = resp.json()
        except Exception as e:
            print(f"    Google request error: {e}")
            break
        if data['status'] not in ['OK', 'ZERO_RESULTS']:
            print(f"    Google status {data['status']} for {keyword}_{lang}: {data.get('error_message', '')}")
            break
        for place in data.get('results', []):
            results.append({
                'place_id': place['place_id'],
                'name': place.get('name', ''),
                'address': place.get('vicinity', ''),
                'lat': place['geometry']['location']['lat'],
                'lng': place['geometry']['location']['lng'],
                'types': ', '.join(place.get('types', [])),
                'keyword': f"{keyword}_{lang}",
                'search_lat': lat,
                'search_lon': lon,
                'rating': place.get('rating'),
                'user_ratings_total': place.get('user_ratings_total', 0),
                'source': 'google',
                'language': lang,
                'category': category
            })
        if 'next_page_token' in data:
            params['pagetoken'] = data['next_page_token']
            time.sleep(2)
        else:
            break
    return results

def save_global_files(results, timestamp):
    """Save cumulative results to CSV, JSON, and GeoPackage (EPSG:3857)."""
    if not results:
        return
    base = os.path.join(OUTPUT_ROOT, f"all_africa_combined_{timestamp}")
    fieldnames = ['place_id','name','address','lat','lng','types','keyword',
                  'search_lat','search_lon','rating','user_ratings_total',
                  'source','language','category']

    # CSV
    with open(f"{base}.csv", 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(results)

    # JSON
    with open(f"{base}.json", 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    # GeoPackage (separate layers)
    try:
        keep_cols = ['place_id','name','address','types','keyword','search_lat','search_lon',
                     'rating','user_ratings_total','source','language','category']
        filling = [r for r in results if r['category'] == 'filling']
        reseller = [r for r in results if r['category'] == 'reseller']
        if filling:
            df = pd.DataFrame(filling)
            geom = [Point(lon, lat) for lon, lat in zip(df['lng'], df['lat'])]
            gdf = gpd.GeoDataFrame(df[[c for c in keep_cols if c in df.columns]],
                                   geometry=geom, crs="EPSG:4326")
            gdf['lat'] = df['lat']
            gdf['lon'] = df['lng']
            gdf.to_crs("EPSG:3857").to_file(f"{base}_filling_3857.gpkg",
                                            driver='GPKG', layer='filling')
        if reseller:
            df = pd.DataFrame(reseller)
            geom = [Point(lon, lat) for lon, lat in zip(df['lng'], df['lat'])]
            gdf = gpd.GeoDataFrame(df[[c for c in keep_cols if c in df.columns]],
                                   geometry=geom, crs="EPSG:4326")
            gdf['lat'] = df['lat']
            gdf['lon'] = df['lng']
            gdf.to_crs("EPSG:3857").to_file(f"{base}_reseller_3857.gpkg",
                                            driver='GPKG', layer='reseller')
    except ImportError:
        pass  # geopandas not installed

# ======================== MAIN PROCESSING ========================

def main():
    if USE_GOOGLE and GOOGLE_API_KEY == "YOUR_VALID_API_KEY":
        print("⚠️  Please insert a valid Google API key in the GOOGLE_API_KEY variable.")
        return

    print("=== LPG Africa Grid Search ===")
    print(f"OSM: {USE_OSM}, Google: {USE_GOOGLE}")
    print(f"Step: {STEP}°, Radius: {RADIUS}m\n")

    # --- 1. Load country geometries and generate grid ---
    print("Loading world boundaries ...")
    country_geoms, shp_crs = load_country_geometries(SSA_COUNTRIES, WORLD_SHP_PATH)
    # We work in EPSG:4326; convert to a projected CRS for buffering
    # Use a simple degree buffer approximation (50 km ≈ 0.45°)
    buffer_deg = 50.0 / 111.0   # approx 0.45045°

    all_raw_points = []   # (lat, lon, country_name)
    print("Generating grid points for selected countries ...")
    for country_name, info in SSA_COUNTRIES.items():
        if country_name not in country_geoms:
            print(f"  Warning: {country_name} not found in shapefile, skipping.")
            continue
        geom = country_geoms[country_name].geometry
        if geom.is_empty:
            continue
        # Buffer the country polygon
        buffered = buffer_geometry(geom, buffer_deg)
        # Generate points
        points = generate_grid(buffered, STEP)
        for (lat, lon) in points:
            all_raw_points.append((lat, lon, country_name))
        print(f"  {country_name}: {len(points)} points inside buffer")

    print(f"\nTotal raw grid points: {len(all_raw_points)}")
    print("Merging duplicate points across borders ...")
    grid_points = deduplicate_grid_points(all_raw_points)
    print(f"Unique search points after merging: {len(grid_points)}")

    # --- 2. Estimate API calls ---
    total_google_calls = 0
    total_osm_points = 0 if USE_OSM else 0
    for (lat, lon, countries) in grid_points:
        langs = languages_for_countries(countries)
        total_google_calls += sum(len(RESELLER_KEYWORDS.get(l, [])) + len(FILLING_KEYWORDS.get(l, []))
                                  for l in langs)
    if USE_OSM:
        total_osm_points = len(grid_points)
    est_google_time = total_google_calls * 0.4  # seconds
    est_osm_time = total_osm_points * 2.5       # rough average with retries/splits
    total_est_sec = est_google_time + est_osm_time
    print(f"\nEstimated API calls: Google {total_google_calls} (~{est_google_time/60:.0f} min)"
          f" + OSM {total_osm_points} (~{est_osm_time/60:.0f} min)"
          f" = ~{total_est_sec/60:.0f} min total\n")

    # --- 3. Load processed points if resuming ---
    processed_set = set()
    if RESUME and os.path.exists(PROCESSED_POINTS_FILE):
        with open(PROCESSED_POINTS_FILE, 'r') as f:
            for line in f:
                line = line.strip()
                if line:
                    lat_str, lon_str = line.split(',')
                    processed_set.add((float(lat_str), float(lon_str)))
        print(f"Resume: {len(processed_set)} points already processed, will skip them.\n")

    # --- 4. Process grid points ---
    global_results = []
    start_time = time.time()
    processed_count = 0

    for idx, (lat, lon, countries) in enumerate(grid_points):
        pt_key = (round(lat, 4), round(lon, 4))
        if pt_key in processed_set:
            continue

        print(f"[{idx+1}/{len(grid_points)}] Processing ({lat}, {lon}) countries: {countries}")

        # Determine languages for this point
        langs = languages_for_countries(countries)
        point_results = []

        # Google search
        if USE_GOOGLE and langs:
            for lang in langs:
                # Filling keywords
                for kw in FILLING_KEYWORDS.get(lang, []):
                    try:
                        places = google_nearby_search(lat, lon, kw, lang, 'filling')
                        point_results.extend(places)
                    except Exception as e:
                        print(f"    Google filling error {kw}_{lang}: {e}")
                    time.sleep(0.4)
                # Reseller keywords
                for kw in RESELLER_KEYWORDS.get(lang, []):
                    try:
                        places = google_nearby_search(lat, lon, kw, lang, 'reseller')
                        point_results.extend(places)
                    except Exception as e:
                        print(f"    Google reseller error {kw}_{lang}: {e}")
                    time.sleep(0.4)

        # OSM search
        if USE_OSM:
            try:
                osm_results = osm_search_adaptive(lat, lon, RADIUS)
                # Update search center for OSM results (they use bbox center, but better to set to actual grid point)
                for r in osm_results:
                    r['search_lat'] = lat
                    r['search_lon'] = lon
                point_results.extend(osm_results)
            except Exception as e:
                print(f"    OSM error: {e}")

        # Deduplicate by place_id, favour filling
        dedup = {}
        for p in point_results:
            pid = p['place_id']
            if pid not in dedup:
                dedup[pid] = p
            elif p['category'] == 'filling' and dedup[pid]['category'] != 'filling':
                dedup[pid] = p
        point_results = list(dedup.values())

        global_results.extend(point_results)

        # Save checkpoint every 50 points or at end
        if (idx + 1) % 50 == 0 or (idx + 1) == len(grid_points):
            save_global_files(global_results, TIMESTAMP)
            with open(PROCESSED_POINTS_FILE, 'w') as f:
                for lat_pt, lon_pt in processed_set:
                    f.write(f"{lat_pt},{lon_pt}\n")
                # Also write current point (already added below)
            print(f"  Checkpoint saved. Cumulative points found: {len(global_results)}")

        # Add to processed set and file
        processed_set.add(pt_key)
        with open(PROCESSED_POINTS_FILE, 'a') as f:
            f.write(f"{pt_key[0]},{pt_key[1]}\n")

        processed_count += 1

    # --- 5. Final save and summary ---
    save_global_files(global_results, TIMESTAMP)
    print(f"\n\nGlobal dataset saved with prefix 'all_africa_combined_{TIMESTAMP}'")

    # Summary CSV per country
    summary_rows = []
    # Count per country
    country_stats = defaultdict(lambda: {'osm': 0, 'google': 0, 'filling': 0, 'reseller': 0, 'lang_counter': Counter()})
    for rec in global_results:
        # We need to assign records to countries; we can use the search point's country set.
        # But records might not have that info directly. Simpler: use original search point's countries_set
        # which we don't keep. Better: For summary we can use the country of the search center, but a record
        # may have been found from multiple points. We'll adopt a simpler approach:
        # Build a mapping from rounded (lat,lng) to countries set, but we already processed by points.
        # We'll re-derive from grid_points: for each result, find which grid points could have generated it?
        # Too complicated. Instead we can just show total numbers without country breakdown.
        # The requirement asks "per ogni paese" but the search is on a grid over the whole region, so
        # a point can belong to multiple countries. We can create summary for the whole region.
        pass

    # Summarise overall (easier and fulfills the core need)
    total_osm = sum(1 for r in global_results if r['source'] == 'osm')
    total_google = len(global_results) - total_osm
    total_filling = sum(1 for r in global_results if r['category'] == 'filling')
    total_reseller = sum(1 for r in global_results if r['category'] == 'reseller')

    summary_rows = [{
        'Region': 'Sub-Saharan Africa',
        'Total_Points': len(global_results),
        'Total_OSM': total_osm,
        'Total_Google': total_google,
        'Total_Filling': total_filling,
        'Total_Reseller': total_reseller,
        'Google_Languages_pct': ''  # could be added
    }]

    summary_path = os.path.join(OUTPUT_ROOT, f"summary_{TIMESTAMP}.csv")
    with open(summary_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['Region','Total_Points','Total_OSM','Total_Google',
                                               'Total_Filling','Total_Reseller','Google_Languages_pct'])
        writer.writeheader()
        writer.writerows(summary_rows)
    print(f"Summary CSV saved: {summary_path}")
    print("Processing completed.")

if __name__ == "__main__":
    main()